# Weight Distributions for the Alpha Fit

When fitting the Morse ``alpha`` parameter for one orientation $(\phi_1, \phi_2)$,
each sampled point can be weighted. This notebook compares the available weight
distributions against the reference energy profile $E(r)$ for a chosen orientation:

- **equal**: every point contributes equally;
- **gaussian**: symmetric weights centred at $r_e$ (width $\sigma$);
- **poisson**: a right-skewed distribution whose maximum sits at $r_e$,
  suppressing the steep repulsive (small-$r$) side without over-damping the
  large-$r$ tail (parameter $\lambda$).

The equilibrium distance $r_e$ is computed with the library's parabola
(quadratic) interpolation via ``extract_energy_minimums``.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from chimorse.config import PLOT_PARAMS, load_molecule_info
from chimorse.datasets import ensure_reference_data
from chimorse.dataio import load_data
from chimorse.analysis import extract_energy_minimums
from chimorse.fitting import (
    equal_weights,
    gaussian_weights,
    poisson_weights,
    energy_weights,
)

plt.rcParams.update(PLOT_PARAMS)

In [ ]:
molecule_name = 'PA'
interaction   = 'OA'
zero_zeta     = True

# Orientation (phi1, phi2) to study, and the weight-distribution parameters.
phi1 = 30
phi2 = 15

gauss_sigmas = [0.5, 1.0]                # Gaussian widths sigma (2 variants)
poisson_lams = [1.0, 0.5]                 # Poisson parameters lam (2 variants)

data_root = Path('../data')
data_dir = ensure_reference_data(molecule_name, data_root=data_root)
molecule = load_molecule_info(molecule_name, metadata_path=data_dir / 'metadata.json')
df = load_data(molecule, interaction, zero_zeta=zero_zeta)

print(f'{molecule.name} {interaction}: {len(df)} rows')
print(f'orientation phi1={phi1}, phi2={phi2}')

In [ ]:
# Radial profile for the chosen orientation.
profile = df[(df['phi1'] == phi1) & (df['phi2'] == phi2)].sort_values('r')


# Equilibrium distance r_e via the library's parabola fit (quadratic interpolation).
emin = extract_energy_minimums(profile, r_max=12, interpolate=True)
re = emin['r'].iloc[0]
e_min = emin['e'].iloc[0]

# Global minimum energy of the whole dataset (used by the energy weight).
e_min_global = df['e'].min()

r = profile['r'].to_numpy()
e = profile['e'].to_numpy()

print(f'r_e = {re:.4f} A   E_min = {e_min:.4f} eV   E_min_global = {e_min_global:.4f} eV  ({len(profile)} sampled r points)')

## Weight distributions alongside the data

Each weight option is shown as a function of $r$, normalised to a peak value of $1$.
The top panel is the reference energy profile $E(r)$ (with $r_e$ marked); the bottom
panel overlays the weight curves on the same radial axis.

In [ ]:
fig, (ax_data, ax_w) = plt.subplots(
    2, 1, figsize=(7, 6), sharex=True,
    gridspec_kw={'height_ratios': [1.5, 1.0]},
)

# ---- reference energy profile ----
ax_data.plot(r, e, '-o', ms=3, lw=1, color='0.35', label=rf'E(r), $\phi_1$={phi1}, $\phi_2$={phi2}')
ax_data.axvline(re, color='r', ls='--', lw=1, label=rf'$r_e$={re:.3f} \AA')
ax_data.axhline(0.0, color='0.6', ls=':', lw=0.8)
ax_data.set_ylabel('binding energy  E (eV)')
ax_data.legend(loc='best', fontsize=8)
ax_data.set_title('Reference E(r) and weight distributions')

# ---- equal weights ----
ax_w.plot(r, equal_weights(r), lw=2, color='k', ls='-', label='equal')

# ---- gaussian weights ----
for sigma in gauss_sigmas:
    w = gaussian_weights(r, re, sigma=sigma)
    ax_w.plot(r, w, lw=1.5, label=rf'gaussian, $\sigma$={sigma:.1f}')

# ---- poisson weights ----
for lam in poisson_lams:
    tag = r'$\lambda = r_e$' if lam is None else rf'$\lambda$={lam:g}'
    w = poisson_weights(r, re, lam=lam)
    ax_w.plot(r, w, lw=1.5, ls='--', label=f'poisson, {tag}')

# ---- energy weights ----
# ---- energy weights (lam variants) ----
for lam in [1/4, 1/2]:
    ew = energy_weights(r, e, re=re, e_min=e_min_global, lam=lam)
    ax_w.plot(r, ew / ew.max(), lw=1.5, ls=':', label=f'energy, lam={lam:g}')

ax_w.axvline(re, color='r', ls='--', lw=1)
ax_w.set_xlabel('intermolecular distance  r (\u00c5)')
ax_w.set_ylabel('weight  w(r)')
ax_w.set_ylim(0, 1.1)
ax_w.legend(loc='best', fontsize=8, ncol=2)

plt.tight_layout()
out_dir = Path('Figures') / 'weights' / molecule.name / interaction
out_dir.mkdir(parents=True, exist_ok=True)
out = out_dir / f'alpha_weights_phi{phi1}_{phi2}.pdf'
plt.savefig(out, bbox_inches='tight', dpi=300)
plt.show()
print(f'saved: {out}')

### Notes

- The **gaussian** weight is symmetric around $r_e$; its width is controlled by $\sigma$.
- The **poisson** weight has its maximum exactly at $r_e$ (mode is placed at $r_e$
  for any choice of $\lambda$) and falls off more steeply toward small $r$
  (the repulsive side) than toward large $r$ (the attractive tail). Smaller
  $\lambda$ increases both the skew and the suppression of the repulsive side.
- These weight functions can be passed to ``fit_alpha_morse`` / ``generate_fourier_morse_data``
  via the ``weight_func`` argument, or selected by name with ``make_weight_func``.